In [1]:
import pandas as pd
from collections import defaultdict
from tqdm import tqdm

In [2]:
DATAFILE = "/mnt/data_storage/datasets/morphology/morphodict-k-forms.csv"

In [3]:
df = pd.read_csv(DATAFILE, sep="\t", index_col=None, names=["form", "segments", "pos", "lemma"])

In [4]:
df.head()

,form,segments,pos,lemma
0,прикопанную,при:PREF/коп:ROOT/а:SUFF/нн:SUFF/ую:END,INFN,прикопать
1,покрытосемянно,по:PREF/кры:PREF/т:SUFF/о:LINK/сем:ROOT/ян:SUF...,ADJF,покрытосемянный
2,переплавившемся,пере:PREF/плав:ROOT/и:SUFF/вш:SUFF/ем:END/ся:POST,INFN,переплавиться
3,вспомянете,вс:PREF/по:PREF/мя:ROOT/н:SUFF/ете:END,INFN,вспомянуть
4,оближет,об:PREF/лиж:ROOT/ет:END,INFN,облизать


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1392790 entries, 0 to 1392789
Data columns (total 4 columns):
 #   Column    Non-Null Count    Dtype
---  ------    --------------    -----
 0   form      1392790 non-null  str  
 1   segments  1392790 non-null  str  
 2   pos       1392790 non-null  str  
 3   lemma     1392790 non-null  str  
dtypes: str(4)
memory usage: 42.5 MB


In [10]:
segments = [s.split("/") for s in df["segments"].tolist()]
markup = [[s.split(":")[1] for s in arr] for arr in segments]

In [16]:
from collections import defaultdict

In [21]:
tag2part = defaultdict(lambda: {"parts": set(), "words": set()})

for segment in segments:
    for part in segment:
        chars, tag = part.split(":")
        tag2part[tag]["parts"].add(chars)
        tag2part[tag]["words"].add("/".join(segment))

In [25]:
for tag, parts in tag2part.items():
    print(tag, len(parts['parts']), sorted(parts['parts'])[:10], list(parts['words'])[:3])

PREF 197 ['а', 'ад', 'ак', 'амби', 'ан', 'ана', 'ано', 'ант', 'анте', 'анти'] ['от:PREF/зол:ROOT/я:SUFF', 'с:PREF/по:PREF/соб:ROOT/ств:SUFF/у:SUFF/ешь:END', 'у:PREF/стро:ROOT/и:SUFF/тел:SUFF/ями:END']
ROOT 7720 ['а', 'абилит', 'аванс', 'авари', 'август', 'ави', 'авиа', 'авось', 'австр', 'австрал'] ['хряст:ROOT/н:SUFF/и:SUFF', 'от:PREF/зол:ROOT/я:SUFF', 'с:PREF/по:PREF/соб:ROOT/ств:SUFF/у:SUFF/ешь:END']
SUFF 729 ['а', 'абель', 'ав', 'авец', 'авиц', 'авич', 'авк', 'авл', 'авник', 'авоч'] ['хряст:ROOT/н:SUFF/и:SUFF', 'от:PREF/зол:ROOT/я:SUFF', 'с:PREF/по:PREF/соб:ROOT/ств:SUFF/у:SUFF/ешь:END']
END 100 ['а', 'ам', 'ами', 'аст', 'ат', 'ах', 'ашь', 'ая', 'го', 'е'] ['с:PREF/по:PREF/соб:ROOT/ств:SUFF/у:SUFF/ешь:END', 'у:PREF/стро:ROOT/и:SUFF/тел:SUFF/ями:END', 'до:PREF/пал:ROOT/ыва:SUFF/вш:SUFF/ее:END/ся:POST']
LINK 11 ['а', 'е', 'ех', 'ехъ', 'и', 'мо', 'н', 'о', 'с', 'т'] ['перв:ROOT/о:LINK/тель:ROOT/н:SUFF/ы:END', 'по:PREF/уз:ROOT/к:SUFF/о:LINK/глаз:ROOT/ей:END', 'перв:ROOT/о:LINK/тель:ROOT

In [ ]:
def prep_append(part: StopAsyncIteration):
    subword, tag = part.split(":")

    if tag == "ROOT":
        result = list(subword)

    else:
        result = [subword]

    return result 

In [38]:
data = []

for segment in segments:
    word = prep_append(segment[0])
    for part in segment[1:]:
        word.extend(prep_append(part))
    word = [word[0]] + ["##" + s for s in word[1:]]
    data.extend(word)

In [39]:
data[:20]

['при',
 '##к',
 '##о',
 '##п',
 '##а',
 '##нн',
 '##ую',
 'по',
 '##кры',
 '##т',
 '##о',
 '##с',
 '##е',
 '##м',
 '##ян',
 '##н',
 '##о',
 'пере',
 '##п',
 '##л']

In [40]:
with open("./segments_split_roots_flat.txt", "w", encoding="utf-8") as file:
    file.write(
        "|".join(data)
    )